In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import pandas as pd
from pathlib import Path


sys.path.append(str(Path.cwd().parent / 'src'))
from model import HousePricePredictor, ModelConfig

In [3]:
data_dir = Path('/home/ayman/SCT_ML_1/data/house-prices-advanced-regression-techniques')
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
display(train_df.head())
display(train_df.info())

In [4]:
def preprocess_data(df):
    # Handle missing values - using ffill() instead of fillna(method='ffill')
    df = df.ffill().fillna(0)
    
    # Convert categorical variables to numerical
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category').cat.codes
    
    return df

# Preprocess the data
X = train_df.drop('SalePrice', axis=1)
y = train_df['SalePrice']
X_processed = preprocess_data(X)

In [5]:
# Initialize config with default values
config = ModelConfig(
    model_type='random_forest',
    random_state=42,
    cv=5,
    verbose=True
)

# Set parameters as single values, not lists
config.param_grids['random_forest']['max_depth'] = 10
config.param_grids['random_forest']['min_samples_split'] = 2
config.param_grids['random_forest']['n_estimators'] = 100  # Set a single value

# Create the model with the updated config
model = HousePricePredictor(config)

# Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

# Train the model
model.train(X_train, y_train)

In [ ]:
predictions = model.predict(X_test, return_confidence=True)

results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': predictions['prediction'],
    'Lower_Bound': predictions['confidence_lower'],  # Changed from 'lower_bound'
    'Upper_Bound': predictions['confidence_upper']   # Changed from 'upper_bound'
})
display(results.head())

In [ ]:
metrics = model.evaluate(X_test, y_test)

In [ ]:
model.plot_feature_importance(top_n=15, interactive=False)


In [ ]:
model.plot_actual_vs_predicted(y_test, predictions['prediction'], interactive=False)

In [ ]:
os.makedirs('models', exist_ok=True)
model.save_model('models/house_price_predictor', save_metadata=True)